# 01 — Explore and Deploy (Pre-Pipeline)

Walk through every step the CI/CD pipeline automates — manually, in your own Snowflake session.

**Prerequisite:** run `sql/setup.sql` in a Snowsight worksheet as ACCOUNTADMIN before starting.

**Sections:**
1. Explore the demo data
2. Verify the semantic view
3. Query the semantic view with natural language
4. Check the agent
5. Chat with the agent
6. Run an evaluation manually
7. Next step — trigger the CI/CD pipeline

## 1 — Explore the Demo Data

In [ ]:
import snowflake.snowpark.context as ctx

session = ctx.get_active_session()

for tbl in ['SIGNUPS', 'TOUCHPOINTS', 'USER_ACTIVITY']:
    n = session.sql(f'SELECT COUNT(*) AS N FROM SV_EVAL_CICD.APP.{tbl}').collect()[0]['N']
    print(f'{tbl}: {n:,} rows')

In [ ]:
kpis = session.sql("""
    SELECT
        COUNT(*) AS total_signups,
        ROUND(SUM(CASE WHEN converted_to_paid THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS conversion_rate_pct,
        ROUND(SUM(CASE WHEN converted_to_paid THEN mrr_amount ELSE 0 END), 2) AS total_mrr
    FROM SV_EVAL_CICD.APP.SIGNUPS
""").to_pandas()
print(kpis.to_string(index=False))

In [ ]:
df = session.sql("""
    SELECT signup_channel,
           COUNT(*) AS signups,
           ROUND(SUM(CASE WHEN converted_to_paid THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS conv_rate_pct
    FROM SV_EVAL_CICD.APP.SIGNUPS
    GROUP BY 1 ORDER BY 2 DESC
""").to_pandas()
print(df.to_string(index=False))

## 2 — Verify the Semantic View

The pipeline deploys `GROWTH_ANALYTICS` from `GROWTH_ANALYTICS.osi.yaml` via `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML`.
After running `setup.sql` the SV does not yet exist — it is created by the first pipeline run.
This cell tells you its current status.

In [ ]:
rows = session.sql("SHOW SEMANTIC VIEWS LIKE 'GROWTH_ANALYTICS' IN SCHEMA SV_EVAL_CICD.APP").collect()
if rows:
    print('Semantic view deployed:', rows[0]['name'])
else:
    print('Semantic view not yet deployed — trigger the CI/CD pipeline (section 7) to deploy it.')

## 3 — Query the Semantic View with Natural Language

Once the SV is deployed, Cortex Analyst translates plain-English questions into SQL.

In [ ]:
import json

questions = [
    'How many signups by channel?',
    'What is the monthly signup trend?',
    'What is our marketing efficiency by channel?',
]

for q in questions:
    payload = json.dumps({
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': q}]}],
        'semantic_model_file': '@SV_EVAL_CICD.APP.CORTEX_STAGE/GROWTH_ANALYTICS.yaml'
    })
    row = session.sql(f"SELECT SNOWFLAKE.CORTEX.ANALYST_TEXT_TO_SQL('{payload}') AS result").collect()[0]
    result = json.loads(row['RESULT'])
    sql_snip = result.get('sql', 'n/a')[:120]
    print(f'Q: {q}')
    print(f'SQL: {sql_snip}...')
    print()

## 4 — Check the Agent

The pipeline creates `GROWTH_AGENT` on first run using `GROWTH_AGENT.agent.yaml`,
then commits a new named version on each subsequent run without disrupting the live default.

In [ ]:
rows = session.sql("SHOW AGENTS LIKE 'GROWTH_AGENT' IN SCHEMA SV_EVAL_CICD.APP").collect()
if rows:
    print('Agent deployed:', rows[0]['name'])
else:
    print('Agent not yet deployed — trigger the pipeline first (section 7).')

## 5 — Chat with the Agent

Ask growth analytics questions. The third question tests boundary enforcement — the agent should refuse.

In [ ]:
import json

def ask(question):
    payload = json.dumps({'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}]})
    row = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
            'SV_EVAL_CICD.APP.GROWTH_AGENT!DEFAULT',
            $${payload}$$
        ) AS response
    """).collect()[0]
    resp = json.loads(row['RESPONSE'])
    msgs = resp.get('messages', [])
    text = msgs[-1].get('content', [{}])[0].get('text', str(resp)) if msgs else str(resp)
    print(f'Q: {question}')
    print(f'A: {text[:300]}')
    print()

ask('How many users signed up in January 2025?')
ask('Which channel had the best conversion rate in Q1 2025?')
ask('What was the weather like in New York on March 1, 2025?')  # should refuse

## 6 — Run an Evaluation Manually

The `eval` job calls `EXECUTE_AI_EVALUATION` against the `GROWTH_AGENT_EVAL` dataset.  
This cell submits one eval run so you can see the score structure.

In [ ]:
import time

run_name = f'notebook-{int(time.time())}'

session.sql(f"""
    CALL SNOWFLAKE.LOCAL.EXECUTE_AI_EVALUATION(
        'SV_EVAL_CICD.APP.GROWTH_AGENT_EVAL',
        OBJECT_CONSTRUCT(
            'agent_fqn', 'SV_EVAL_CICD.APP.GROWTH_AGENT!DEFAULT',
            'run_name',  '{run_name}'
        )
    )
""").collect()

print(f'Eval submitted — run name: {run_name}')
print('Wait 2-3 minutes, then run the next cell to see scores.')

In [ ]:
# Run after the eval above completes
scores = session.sql(f"""
    SELECT metric_name, ROUND(AVG(eval_agg_score), 3) AS avg_score
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'SV_EVAL_CICD', 'APP', 'GROWTH_AGENT', 'CORTEX AGENT', '{run_name}'
    ))
    GROUP BY 1 ORDER BY 1
""").to_pandas()
print(scores.to_string(index=False))

## 7 — Next Step: Trigger the CI/CD Pipeline

You have now run every step manually. The CI/CD pipeline automates all of it on every push to `main`.

**To trigger the pipeline:**
1. Go to your fork of `semantic-view-eval-cicd` on GitHub
2. Click the **Actions** tab
3. Select **Deploy and evaluate Cortex project**
4. Click **Run workflow → Run workflow**

The five jobs take ~8 minutes. When the run is green, open
`02_Inspect_Pipeline_Results.ipynb` to inspect what the pipeline produced.